In [8]:
import spacy

print("spaCy version:", spacy.__version__)
print("spaCy is working!")

spaCy version: 3.8.7
spaCy is working!


In [9]:
import pandas as pd

# Load our BIO dataset
df = pd.read_csv("../data/model4_ner_bio_dataset.csv")

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 10 rows:")
display(df.head(10))

Dataset shape: (40830, 3)

Columns: ['transcript_id', 'token', 'label']

First 10 rows:


,transcript_id,token,label
0,T0001,I,O
1,T0001,am,O
2,T0001,a,O
3,T0001,60,B-AGE
4,T0001,year,O
5,T0001,old,O
6,T0001,male,B-GENDER
7,T0001,berojgar,B-OCCUPATION
8,T0001,living,O
9,T0001,in,O


In [10]:
import spacy

# Group tokens back into complete transcripts
grouped = df.groupby("transcript_id")

training_data = []

for transcript_id, group in grouped:
    tokens = group["token"].tolist()
    labels = group["label"].tolist()

    # Reconstruct the original text
    text = " ".join(tokens)

    entities = []
    current_entity = None

    # Character position tracker
    char_pos = 0

    for token, label in zip(tokens, labels):
        start = char_pos
        end = start + len(token)

        if label.startswith("B-"):
            # Save previous entity
            if current_entity is not None:
                entities.append(current_entity)

            entity_type = label[2:]
            current_entity = [start, end, entity_type]

        elif label.startswith("I-"):
            if current_entity is not None:
                # Extend the current entity
                current_entity[1] = end

        else:
            # Save previous entity
            if current_entity is not None:
                entities.append(current_entity)
                current_entity = None

        # +1 for the space between tokens
        char_pos = end + 1

    # Save final entity
    if current_entity is not None:
        entities.append(current_entity)

    training_data.append((text, {"entities": entities}))

print("Number of transcripts:", len(training_data))
print("\nFirst training example:")
print(training_data[0])

Number of transcripts: 2000

First training example:
('I am a 60 year old male berojgar living in Madhya Pradesh My annual household income is 120000 rupees', {'entities': [[7, 9, 'AGE'], [19, 23, 'GENDER'], [24, 32, 'OCCUPATION'], [43, 57, 'STATE'], [88, 94, 'ANNUAL_INCOME']]})


In [11]:
from spacy.training import Example

nlp = spacy.blank("en")

valid_examples = []
invalid_examples = []

for text, annotations in training_data:
    doc = nlp.make_doc(text)

    try:
        example = Example.from_dict(
            doc,
            annotations
        )
        valid_examples.append(example)

    except Exception as e:
        invalid_examples.append((text, str(e)))

print("Total examples:", len(training_data))
print("Valid examples:", len(valid_examples))
print("Invalid examples:", len(invalid_examples))

if invalid_examples:
    print("\nFirst invalid example:")
    print(invalid_examples[0])

Total examples: 2000
Valid examples: 2000
Invalid examples: 0


In [12]:
from sklearn.model_selection import train_test_split

# Split transcript-level examples
train_data, temp_data = train_test_split(
    training_data,
    test_size=0.20,
    random_state=42
)

# Split remaining 20% into validation and test
val_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42
)

print("Total:", len(training_data))
print("Training:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

Total: 2000
Training: 1600
Validation: 200
Test: 200


In [13]:


# Create a blank English pipeline
nlp = spacy.blank("en")

# Add NER component
ner = nlp.add_pipe("ner")

# Our custom entity labels
ENTITY_LABELS = [
    "AGE",
    "GENDER",
    "OCCUPATION",
    "STATE",
    "ANNUAL_INCOME",
    "CASTE_CATEGORY",
    "DISABILITY"
]

# Add labels to the NER component
for label in ENTITY_LABELS:
    ner.add_label(label)

print("Pipeline:", nlp.pipe_names)
print("Entity labels:", ner.labels)

Pipeline: ['ner']
Entity labels: ('AGE', 'ANNUAL_INCOME', 'CASTE_CATEGORY', 'DISABILITY', 'GENDER', 'OCCUPATION', 'STATE')


In [14]:
import random
from spacy.util import minibatch, compounding
from spacy.training import Example

# Disable other pipeline components during training
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]

# Initialize the model
with nlp.disable_pipes(*other_pipes):
    optimizer = nlp.begin_training()

    print("Starting training...\n")

    for iteration in range(30):
        random.shuffle(train_data)
        losses = {}

        # Create batches
        batches = minibatch(
            train_data,
            size=compounding(4.0, 32.0, 1.5)
        )

        for batch in batches:
            examples = []

            for text, annotations in batch:
                doc = nlp.make_doc(text)

                example = Example.from_dict(
                    doc,
                    annotations
                )

                examples.append(example)

            nlp.update(
                examples,
                drop=0.3,
                losses=losses
            )

        print(
            f"Iteration {iteration + 1}/30 "
            f"Loss: {losses.get('ner', 0):.4f}"
        )

print("\nTraining completed!")

Starting training...

Iteration 1/30 Loss: 9228.0967
Iteration 2/30 Loss: 258.1973
Iteration 3/30 Loss: 55.0061
Iteration 4/30 Loss: 62.2582
Iteration 5/30 Loss: 60.0191
Iteration 6/30 Loss: 59.5191
Iteration 7/30 Loss: 45.6226
Iteration 8/30 Loss: 47.4257
Iteration 9/30 Loss: 47.5652
Iteration 10/30 Loss: 47.0170
Iteration 11/30 Loss: 44.4404
Iteration 12/30 Loss: 47.0542
Iteration 13/30 Loss: 48.7837
Iteration 14/30 Loss: 47.2326
Iteration 15/30 Loss: 47.1624
Iteration 16/30 Loss: 53.8527
Iteration 17/30 Loss: 44.9774
Iteration 18/30 Loss: 48.3417
Iteration 19/30 Loss: 46.1759
Iteration 20/30 Loss: 46.2314
Iteration 21/30 Loss: 47.2860
Iteration 22/30 Loss: 43.3511
Iteration 23/30 Loss: 53.2132
Iteration 24/30 Loss: 47.0042
Iteration 25/30 Loss: 49.2225
Iteration 26/30 Loss: 46.1425
Iteration 27/30 Loss: 43.5663
Iteration 28/30 Loss: 49.2955
Iteration 29/30 Loss: 54.4357
Iteration 30/30 Loss: 45.1461

Training completed!


In [15]:
from spacy.training import Example

def evaluate_ner(nlp, data):
    examples = []

    for text, annotations in data:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        examples.append(example)

    scores = nlp.evaluate(examples)

    return scores


# Evaluate on the test set
test_scores = evaluate_ner(nlp, test_data)

print("NER Evaluation Results")
print("=" * 40)

print("Precision :", round(test_scores["ents_p"], 4))
print("Recall    :", round(test_scores["ents_r"], 4))
print("F1 Score  :", round(test_scores["ents_f"], 4))

NER Evaluation Results
Precision : 1.0
Recall    : 0.998
F1 Score  : 0.999


In [16]:
def test_ner(text):
    doc = nlp(text)

    print("\nInput:")
    print(text)

    print("\nExtracted entities:")
    if doc.ents:
        for ent in doc.ents:
            print(f"{ent.text:25} → {ent.label_}")
    else:
        print("No entities detected")


# Test 1
test_ner(
    "Meri age 22 saal hai, main student hoon aur Uttar Pradesh mein rehti hoon."
)

# Test 2
test_ner(
    "I am a 45 year old farmer from Maharashtra with annual income of 250000."
)

# Test 3
test_ner(
    "Main OBC category se hoon aur meri disability hai."
)


Input:
Meri age 22 saal hai, main student hoon aur Uttar Pradesh mein rehti hoon.

Extracted entities:
22                        → AGE
student                   → OCCUPATION
Uttar Pradesh             → STATE

Input:
I am a 45 year old farmer from Maharashtra with annual income of 250000.

Extracted entities:
45                        → AGE
Maharashtra               → STATE
250000                    → ANNUAL_INCOME

Input:
Main OBC category se hoon aur meri disability hai.

Extracted entities:
OBC                       → CASTE_CATEGORY
disability                → AGE


In [17]:
#Model Improvement
from collections import Counter

entity_counts = Counter()

for text, annotations in train_data:
    for start, end, label in annotations["entities"]:
        entity_counts[label] += 1

print("Training entity counts:")
print("=" * 35)

for label, count in entity_counts.most_common():
    print(f"{label:20} : {count}")

Training entity counts:
STATE                : 1600
ANNUAL_INCOME        : 1600
OCCUPATION           : 1352
AGE                  : 1343
CASTE_CATEGORY       : 800
DISABILITY           : 543
GENDER               : 522


In [18]:
from collections import Counter

occupation_values = Counter()
disability_values = Counter()

for text, annotations in train_data:
    for start, end, label in annotations["entities"]:
        entity_text = text[start:end].strip().lower()

        if label == "OCCUPATION":
            occupation_values[entity_text] += 1

        elif label == "DISABILITY":
            disability_values[entity_text] += 1

print("Top OCCUPATION examples:")
print("=" * 40)

for entity, count in occupation_values.most_common(30):
    print(f"{entity:25} : {count}")

print("\nDISABILITY examples:")
print("=" * 40)

for entity, count in disability_values.most_common(30):
    print(f"{entity:25} : {count}")

Top OCCUPATION examples:
student                   : 115
berojgar                  : 109
street vendor             : 107
daily wage worker         : 105
unemployed                : 100
bunkar                    : 99
farmer                    : 94
chhatra                   : 92
kisan                     : 92
labour                    : 92
weaver                    : 91
rehri wala                : 89
mazdoor                   : 84
artisan                   : 83

DISABILITY examples:
divyang                   : 273
differently abled         : 270


In [19]:
# Targeted augmentation examples for better generalization

augmentation_data = [
    (
        "I am a farmer from Uttar Pradesh",
        {"entities": [
            (9, 15, "OCCUPATION"),
            (21, 35, "STATE")
        ]}
    ),
    (
        "My occupation is farmer and I live in Maharashtra",
        {"entities": [
            (18, 24, "OCCUPATION"),
            (45, 55, "STATE")
        ]}
    ),
    (
        "I work as a farmer in Madhya Pradesh",
        {"entities": [
            (13, 19, "OCCUPATION"),
            (23, 37, "STATE")
        ]}
    ),
    (
        "Main farmer hoon aur Uttar Pradesh mein rehta hoon",
        {"entities": [
            (5, 11, "OCCUPATION"),
            (21, 35, "STATE")
        ]}
    ),
    (
        "Meri disability hai",
        {"entities": [
            (5, 14, "DISABILITY")
        ]}
    ),
    (
        "I have a disability",
        {"entities": [
            (9, 19, "DISABILITY")
        ]}
    ),
    (
        "Main differently abled hoon",
        {"entities": [
            (5, 21, "DISABILITY")
        ]}
    ),
    (
        "Main divyang hoon",
        {"entities": [
            (5, 12, "DISABILITY")
        ]}
    ),
    (
        "Meri age 22 hai, main student hoon",
        {"entities": [
            (9, 11, "AGE"),
            (23, 30, "OCCUPATION")
        ]}
    ),
    (
        "I am a student, from Uttar Pradesh",
        {"entities": [
            (7, 14, "OCCUPATION"),
            (21, 35, "STATE")
        ]}
    )
]

print("Augmentation examples:", len(augmentation_data))

Augmentation examples: 10


In [20]:
from spacy.training import Example

augmentation_examples = []

for text, annotations in augmentation_data:
    doc = nlp.make_doc(text)

    try:
        example = Example.from_dict(doc, annotations)
        augmentation_examples.append(example)

    except Exception as e:
        print("Invalid example:")
        print(text)
        print("Error:", e)

print("Total augmentation examples:", len(augmentation_data))
print("Valid augmentation examples:", len(augmentation_examples))

Total augmentation examples: 10
Valid augmentation examples: 10


c:\Users\hp\OneDrive\Desktop\Entity Extraction\venv\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "I am a farmer from Uttar Pradesh" with entities "[(9, 15, 'OCCUPATION'), (21, 35, 'STATE')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
c:\Users\hp\OneDrive\Desktop\Entity Extraction\venv\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "My occupation is farmer and I live in Maharashtra" with entities "[(18, 24, 'OCCUPATION'), (45, 55, 'STATE')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
c:\Users\hp\OneDrive\Desktop\Entity Extraction\venv\lib\site-packages\spacy\training\iob_utils.py:149: Use

In [21]:
def find_entity(text, entity_text, label):
    """
    Automatically find the character span of an entity.
    """
    start = text.lower().find(entity_text.lower())

    if start == -1:
        raise ValueError(
            f"Could not find '{entity_text}' in: {text}"
        )

    end = start + len(entity_text)

    return (start, end, label)


# Rebuild augmentation data with automatic offsets

augmentation_raw = [
    (
        "I am a farmer from Uttar Pradesh",
        [
            ("farmer", "OCCUPATION"),
            ("Uttar Pradesh", "STATE")
        ]
    ),
    (
        "My occupation is farmer and I live in Maharashtra",
        [
            ("farmer", "OCCUPATION"),
            ("Maharashtra", "STATE")
        ]
    ),
    (
        "I work as a farmer in Madhya Pradesh",
        [
            ("farmer", "OCCUPATION"),
            ("Madhya Pradesh", "STATE")
        ]
    ),
    (
        "Main farmer hoon aur Uttar Pradesh mein rehta hoon",
        [
            ("farmer", "OCCUPATION"),
            ("Uttar Pradesh", "STATE")
        ]
    ),
    (
        "Meri disability hai",
        [
            ("disability", "DISABILITY")
        ]
    ),
    (
        "I have a disability",
        [
            ("disability", "DISABILITY")
        ]
    ),
    (
        "Main differently abled hoon",
        [
            ("differently abled", "DISABILITY")
        ]
    ),
    (
        "Main divyang hoon",
        [
            ("divyang", "DISABILITY")
        ]
    ),
    (
        "Meri age 22 hai, main student hoon",
        [
            ("22", "AGE"),
            ("student", "OCCUPATION")
        ]
    ),
    (
        "I am a student, from Uttar Pradesh",
        [
            ("student", "OCCUPATION"),
            ("Uttar Pradesh", "STATE")
        ]
    )
]


augmentation_data_fixed = []

for text, entities in augmentation_raw:

    spans = []

    for entity_text, label in entities:
        span = find_entity(text, entity_text, label)
        spans.append(span)

    augmentation_data_fixed.append(
        (text, {"entities": spans})
    )


print("Fixed augmentation examples:", len(augmentation_data_fixed))

for item in augmentation_data_fixed:
    print(item)

Fixed augmentation examples: 10
('I am a farmer from Uttar Pradesh', {'entities': [(7, 13, 'OCCUPATION'), (19, 32, 'STATE')]})
('My occupation is farmer and I live in Maharashtra', {'entities': [(17, 23, 'OCCUPATION'), (38, 49, 'STATE')]})
('I work as a farmer in Madhya Pradesh', {'entities': [(12, 18, 'OCCUPATION'), (22, 36, 'STATE')]})
('Main farmer hoon aur Uttar Pradesh mein rehta hoon', {'entities': [(5, 11, 'OCCUPATION'), (21, 34, 'STATE')]})
('Meri disability hai', {'entities': [(5, 15, 'DISABILITY')]})
('I have a disability', {'entities': [(9, 19, 'DISABILITY')]})
('Main differently abled hoon', {'entities': [(5, 22, 'DISABILITY')]})
('Main divyang hoon', {'entities': [(5, 12, 'DISABILITY')]})
('Meri age 22 hai, main student hoon', {'entities': [(9, 11, 'AGE'), (22, 29, 'OCCUPATION')]})
('I am a student, from Uttar Pradesh', {'entities': [(7, 14, 'OCCUPATION'), (21, 34, 'STATE')]})


In [22]:
augmentation_examples = []

for text, annotations in augmentation_data_fixed:
    doc = nlp.make_doc(text)

    example = Example.from_dict(
        doc,
        annotations
    )

    augmentation_examples.append(example)

print("Total augmentation examples:", len(augmentation_data_fixed))
print("Valid augmentation examples:", len(augmentation_examples))

Total augmentation examples: 10
Valid augmentation examples: 10


In [23]:
# Combine original training data with our targeted augmentation
improved_train_data = train_data + augmentation_data_fixed

print("Original training examples:", len(train_data))
print("Augmentation examples:", len(augmentation_data_fixed))
print("Improved training examples:", len(improved_train_data))

Original training examples: 1600
Augmentation examples: 10
Improved training examples: 1610


In [24]:
import random
from spacy.training import Example
from spacy.util import minibatch, compounding

# Train only the NER component
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]

with nlp.disable_pipes(*other_pipes):

    optimizer = nlp.begin_training()

    print("Starting improved training...\n")

    for iteration in range(30):

        random.shuffle(improved_train_data)
        losses = {}

        batches = minibatch(
            improved_train_data,
            size=compounding(4.0, 32.0, 1.5)
        )

        for batch in batches:

            examples = []

            for text, annotations in batch:

                doc = nlp.make_doc(text)

                example = Example.from_dict(
                    doc,
                    annotations
                )

                examples.append(example)

            nlp.update(
                examples,
                drop=0.3,
                losses=losses
            )

        print(
            f"Iteration {iteration + 1}/30 "
            f"Loss: {losses.get('ner', 0):.4f}"
        )

print("\nImproved training completed!")

Starting improved training...

Iteration 1/30 Loss: 9237.1777
Iteration 2/30 Loss: 301.2990
Iteration 3/30 Loss: 72.2282
Iteration 4/30 Loss: 58.4525
Iteration 5/30 Loss: 50.4475
Iteration 6/30 Loss: 47.1989
Iteration 7/30 Loss: 54.8420
Iteration 8/30 Loss: 44.8971
Iteration 9/30 Loss: 53.8159
Iteration 10/30 Loss: 50.6886
Iteration 11/30 Loss: 50.8671
Iteration 12/30 Loss: 46.7037
Iteration 13/30 Loss: 48.3774
Iteration 14/30 Loss: 47.7648
Iteration 15/30 Loss: 48.5808
Iteration 16/30 Loss: 55.0226
Iteration 17/30 Loss: 48.1566
Iteration 18/30 Loss: 49.3912
Iteration 19/30 Loss: 58.1672
Iteration 20/30 Loss: 47.6064
Iteration 21/30 Loss: 47.6983
Iteration 22/30 Loss: 49.0506
Iteration 23/30 Loss: 47.5954
Iteration 24/30 Loss: 49.0543
Iteration 25/30 Loss: 45.5247
Iteration 26/30 Loss: 51.3699
Iteration 27/30 Loss: 49.6074
Iteration 28/30 Loss: 56.0603
Iteration 29/30 Loss: 48.6574
Iteration 30/30 Loss: 45.1742

Improved training completed!


In [25]:
def test_ner(text):
    doc = nlp(text)

    print("\nInput:")
    print(text)

    print("\nExtracted entities:")
    if doc.ents:
        for ent in doc.ents:
            print(f"{ent.text:25} → {ent.label_}")
    else:
        print("No entities detected")


# Test 1: Hinglish
test_ner(
    "Meri age 22 saal hai, main student hoon aur Uttar Pradesh mein rehti hoon."
)

# Test 2: English
test_ner(
    "I am a 45 year old farmer from Maharashtra with annual income of 250000."
)

# Test 3: Caste + disability
test_ner(
    "Main OBC category se hoon aur meri disability hai."
)


Input:
Meri age 22 saal hai, main student hoon aur Uttar Pradesh mein rehti hoon.

Extracted entities:
22                        → AGE
student                   → OCCUPATION
Uttar Pradesh             → STATE

Input:
I am a 45 year old farmer from Maharashtra with annual income of 250000.

Extracted entities:
45                        → AGE
Maharashtra               → STATE
250000                    → ANNUAL_INCOME

Input:
Main OBC category se hoon aur meri disability hai.

Extracted entities:
OBC                       → CASTE_CATEGORY
disability                → DISABILITY


In [26]:
farmer_augmentation = [
    (
        "I am a farmer",
        {"entities": [(7, 13, "OCCUPATION")]}
    ),
    (
        "I work as a farmer",
        {"entities": [(12, 18, "OCCUPATION")]}
    ),
    (
        "My occupation is farmer",
        {"entities": [(18, 24, "OCCUPATION")]}
    ),
    (
        "I am a farmer from Maharashtra",
        {"entities": [
            (7, 13, "OCCUPATION"),
            (19, 30, "STATE")
        ]}
    ),
    (
        "I am a farmer from Uttar Pradesh",
        {"entities": [
            (7, 13, "OCCUPATION"),
            (19, 32, "STATE")
        ]}
    ),
    (
        "I am a farmer with annual income of 250000",
        {"entities": [
            (7, 13, "OCCUPATION"),
            (39, 45, "ANNUAL_INCOME")
        ]}
    ),
    (
        "Main ek farmer hoon",
        {"entities": [(9, 15, "OCCUPATION")]}
    ),
    (
        "Main farmer hoon",
        {"entities": [(5, 11, "OCCUPATION")]}
    ),
    (
        "Mera occupation farmer hai",
        {"entities": [(15, 21, "OCCUPATION")]}
    ),
    (
        "Farmer by occupation",
        {"entities": [(0, 6, "OCCUPATION")]}
    )
]

print("Farmer augmentation examples:", len(farmer_augmentation))

Farmer augmentation examples: 10


In [27]:
from spacy.training import Example

valid_farmer_examples = []

for text, annotations in farmer_augmentation:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annotations)
    valid_farmer_examples.append((text, annotations))

print("Total farmer augmentation examples:", len(farmer_augmentation))
print("Valid farmer augmentation examples:", len(valid_farmer_examples))

print("\nExamples:")
for text, annotations in valid_farmer_examples:
    print(text)
    print(annotations)
    print()

Total farmer augmentation examples: 10
Valid farmer augmentation examples: 10

Examples:
I am a farmer
{'entities': [(7, 13, 'OCCUPATION')]}

I work as a farmer
{'entities': [(12, 18, 'OCCUPATION')]}

My occupation is farmer
{'entities': [(18, 24, 'OCCUPATION')]}

I am a farmer from Maharashtra
{'entities': [(7, 13, 'OCCUPATION'), (19, 30, 'STATE')]}

I am a farmer from Uttar Pradesh
{'entities': [(7, 13, 'OCCUPATION'), (19, 32, 'STATE')]}

I am a farmer with annual income of 250000
{'entities': [(7, 13, 'OCCUPATION'), (39, 45, 'ANNUAL_INCOME')]}

Main ek farmer hoon
{'entities': [(9, 15, 'OCCUPATION')]}

Main farmer hoon
{'entities': [(5, 11, 'OCCUPATION')]}

Mera occupation farmer hai
{'entities': [(15, 21, 'OCCUPATION')]}

Farmer by occupation
{'entities': [(0, 6, 'OCCUPATION')]}



c:\Users\hp\OneDrive\Desktop\Entity Extraction\venv\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "My occupation is farmer" with entities "[(18, 24, 'OCCUPATION')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
c:\Users\hp\OneDrive\Desktop\Entity Extraction\venv\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "I am a farmer with annual income of 250000" with entities "[(7, 13, 'OCCUPATION'), (39, 45, 'ANNUAL_INCOME')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
c:\Users\hp\OneDrive\Desktop\Entity Extraction\venv\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entit

In [28]:
def find_entity(text, entity_text, label):
    start = text.lower().find(entity_text.lower())

    if start == -1:
        raise ValueError(
            f"Could not find '{entity_text}' in: {text}"
        )

    end = start + len(entity_text)
    return (start, end, label)


farmer_augmentation_raw = [
    (
        "I am a farmer",
        [("farmer", "OCCUPATION")]
    ),
    (
        "I work as a farmer",
        [("farmer", "OCCUPATION")]
    ),
    (
        "My occupation is farmer",
        [("farmer", "OCCUPATION")]
    ),
    (
        "I am a farmer from Maharashtra",
        [
            ("farmer", "OCCUPATION"),
            ("Maharashtra", "STATE")
        ]
    ),
    (
        "I am a farmer from Uttar Pradesh",
        [
            ("farmer", "OCCUPATION"),
            ("Uttar Pradesh", "STATE")
        ]
    ),
    (
        "I am a farmer with annual income of 250000",
        [
            ("farmer", "OCCUPATION"),
            ("250000", "ANNUAL_INCOME")
        ]
    ),
    (
        "Main ek farmer hoon",
        [("farmer", "OCCUPATION")]
    ),
    (
        "Main farmer hoon",
        [("farmer", "OCCUPATION")]
    ),
    (
        "Mera occupation farmer hai",
        [("farmer", "OCCUPATION")]
    ),
    (
        "Farmer by occupation",
        [("Farmer", "OCCUPATION")]
    )
]


farmer_augmentation_fixed = []

for text, entities in farmer_augmentation_raw:
    spans = []

    for entity_text, label in entities:
        span = find_entity(text, entity_text, label)
        spans.append(span)

    farmer_augmentation_fixed.append(
        (text, {"entities": spans})
    )


print("Fixed farmer augmentation:", len(farmer_augmentation_fixed))

for item in farmer_augmentation_fixed:
    print(item)

Fixed farmer augmentation: 10
('I am a farmer', {'entities': [(7, 13, 'OCCUPATION')]})
('I work as a farmer', {'entities': [(12, 18, 'OCCUPATION')]})
('My occupation is farmer', {'entities': [(17, 23, 'OCCUPATION')]})
('I am a farmer from Maharashtra', {'entities': [(7, 13, 'OCCUPATION'), (19, 30, 'STATE')]})
('I am a farmer from Uttar Pradesh', {'entities': [(7, 13, 'OCCUPATION'), (19, 32, 'STATE')]})
('I am a farmer with annual income of 250000', {'entities': [(7, 13, 'OCCUPATION'), (36, 42, 'ANNUAL_INCOME')]})
('Main ek farmer hoon', {'entities': [(8, 14, 'OCCUPATION')]})
('Main farmer hoon', {'entities': [(5, 11, 'OCCUPATION')]})
('Mera occupation farmer hai', {'entities': [(16, 22, 'OCCUPATION')]})
('Farmer by occupation', {'entities': [(0, 6, 'OCCUPATION')]})


In [29]:
from spacy.training import Example

valid_farmer_examples = []

for text, annotations in farmer_augmentation_fixed:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annotations)
    valid_farmer_examples.append(example)

print("Total farmer augmentation examples:", len(farmer_augmentation_fixed))
print("Valid farmer augmentation examples:", len(valid_farmer_examples))

Total farmer augmentation examples: 10
Valid farmer augmentation examples: 10


In [30]:
improved_train_data_v3 = (
    train_data
    + augmentation_data_fixed
    + farmer_augmentation_fixed
)

print("Original training examples:", len(train_data))
print("Previous augmentation:", len(augmentation_data_fixed))
print("Farmer augmentation:", len(farmer_augmentation_fixed))
print("Total Version 3 training examples:", len(improved_train_data_v3))

Original training examples: 1600
Previous augmentation: 10
Farmer augmentation: 10
Total Version 3 training examples: 1620


In [31]:


# Create a fresh blank model
nlp_v3 = spacy.blank("en")

# Add NER pipeline
ner_v3 = nlp_v3.add_pipe("ner")

# Add all entity labels
for label in ENTITY_LABELS:
    ner_v3.add_label(label)

print("Pipeline:", nlp_v3.pipe_names)
print("Entity labels:", ner_v3.labels)

Pipeline: ['ner']
Entity labels: ('AGE', 'ANNUAL_INCOME', 'CASTE_CATEGORY', 'DISABILITY', 'GENDER', 'OCCUPATION', 'STATE')


In [32]:

from spacy.training import Example
from spacy.util import minibatch, compounding

other_pipes = [pipe for pipe in nlp_v3.pipe_names if pipe != "ner"]

with nlp_v3.disable_pipes(*other_pipes):
    optimizer = nlp_v3.begin_training()

    print("Starting Model 4 Version 3 training...\n")

    for iteration in range(30):
        random.shuffle(improved_train_data_v3)
        losses = {}

        batches = minibatch(
            improved_train_data_v3,
            size=compounding(4.0, 32.0, 1.5)
        )

        for batch in batches:
            examples = []

            for text, annotations in batch:
                doc = nlp_v3.make_doc(text)
                example = Example.from_dict(doc, annotations)
                examples.append(example)

            nlp_v3.update(
                examples,
                drop=0.3,
                losses=losses
            )

        print(
            f"Iteration {iteration + 1}/30 "
            f"Loss: {losses.get('ner', 0):.4f}"
        )

print("\nModel 4 Version 3 training completed!")

Starting Model 4 Version 3 training...

Iteration 1/30 Loss: 9562.6543
Iteration 2/30 Loss: 293.1242
Iteration 3/30 Loss: 58.9725
Iteration 4/30 Loss: 68.7565
Iteration 5/30 Loss: 56.4961
Iteration 6/30 Loss: 50.1925
Iteration 7/30 Loss: 51.3245
Iteration 8/30 Loss: 51.0215
Iteration 9/30 Loss: 50.2090
Iteration 10/30 Loss: 51.4448
Iteration 11/30 Loss: 47.7811
Iteration 12/30 Loss: 51.6437
Iteration 13/30 Loss: 67.7300
Iteration 14/30 Loss: 45.0471
Iteration 15/30 Loss: 45.5668
Iteration 16/30 Loss: 50.7832
Iteration 17/30 Loss: 46.8079
Iteration 18/30 Loss: 47.5705
Iteration 19/30 Loss: 50.2170
Iteration 20/30 Loss: 49.0519
Iteration 21/30 Loss: 47.9354
Iteration 22/30 Loss: 48.5225
Iteration 23/30 Loss: 45.2188
Iteration 24/30 Loss: 47.5824
Iteration 25/30 Loss: 42.8229
Iteration 26/30 Loss: 58.8323
Iteration 27/30 Loss: 47.3230
Iteration 28/30 Loss: 49.2035
Iteration 29/30 Loss: 45.7861
Iteration 30/30 Loss: 49.7812

Model 4 Version 3 training completed!


In [33]:
def test_ner_v3(text):
    doc = nlp_v3(text)

    print("\nInput:")
    print(text)

    print("\nExtracted entities:")
    if doc.ents:
        for ent in doc.ents:
            print(f"{ent.text:25} → {ent.label_}")
    else:
        print("No entities detected")


# Test 1
test_ner_v3(
    "Meri age 22 saal hai, main student hoon aur Uttar Pradesh mein rehti hoon."
)

# Test 2
test_ner_v3(
    "I am a 45 year old farmer from Maharashtra with annual income of 250000."
)

# Test 3
test_ner_v3(
    "Main OBC category se hoon aur meri disability hai."
)


Input:
Meri age 22 saal hai, main student hoon aur Uttar Pradesh mein rehti hoon.

Extracted entities:
22                        → AGE
student                   → OCCUPATION
Uttar Pradesh             → STATE
rehti                     → GENDER

Input:
I am a 45 year old farmer from Maharashtra with annual income of 250000.

Extracted entities:
45                        → AGE
Maharashtra               → STATE
250000                    → ANNUAL_INCOME

Input:
Main OBC category se hoon aur meri disability hai.

Extracted entities:
OBC                       → CASTE_CATEGORY
disability                → DISABILITY


In [34]:
from spacy.training import Example
from spacy.scorer import Scorer

examples = []

for text, annotations in test_data:
    doc = nlp_v3.make_doc(text)
    example = Example.from_dict(doc, annotations)
    examples.append(example)

scorer = Scorer()
scores = scorer.score(examples)

print("Precision:", round(scores["ents_p"], 4))
print("Recall:", round(scores["ents_r"], 4))
print("F1 Score:", round(scores["ents_f"], 4))

print("\nPer-entity scores:")
for label, metrics in scores["ents_per_type"].items():
    print(
        f"{label:20} "
        f"P={metrics['p']:.4f} "
        f"R={metrics['r']:.4f} "
        f"F1={metrics['f']:.4f}"
    )

Precision: 0.0
Recall: 0.0
F1 Score: 0.0

Per-entity scores:
AGE                  P=0.0000 R=0.0000 F1=0.0000
STATE                P=0.0000 R=0.0000 F1=0.0000
ANNUAL_INCOME        P=0.0000 R=0.0000 F1=0.0000
CASTE_CATEGORY       P=0.0000 R=0.0000 F1=0.0000
OCCUPATION           P=0.0000 R=0.0000 F1=0.0000
DISABILITY           P=0.0000 R=0.0000 F1=0.0000
GENDER               P=0.0000 R=0.0000 F1=0.0000


In [35]:
print("Number of test examples:", len(test_data))

print("\nFirst test example:")
print(test_data[0])

print("\nType of first test example:")
print(type(test_data[0]))

print("\nNumber of entities in first example:")
print(test_data[0][1]["entities"])

Number of test examples: 200

First test example:
('I belong to SC category from Bihar Age is 20 profession is berojgar income around 80000', {'entities': [[12, 14, 'CASTE_CATEGORY'], [29, 34, 'STATE'], [42, 44, 'AGE'], [59, 67, 'OCCUPATION'], [82, 87, 'ANNUAL_INCOME']]})

Type of first test example:
<class 'tuple'>

Number of entities in first example:
[[12, 14, 'CASTE_CATEGORY'], [29, 34, 'STATE'], [42, 44, 'AGE'], [59, 67, 'OCCUPATION'], [82, 87, 'ANNUAL_INCOME']]


In [36]:
from spacy.training import Example
from spacy.scorer import Scorer

examples = []

for text, annotations in test_data:
    pred_doc = nlp_v3(text)
    example = Example.from_dict(pred_doc, annotations)
    examples.append(example)

scorer = Scorer()
scores = scorer.score(examples)

print("Precision:", round(scores["ents_p"], 4))
print("Recall:", round(scores["ents_r"], 4))
print("F1 Score:", round(scores["ents_f"], 4))

print("\nPer-entity scores:")

for label, metrics in scores["ents_per_type"].items():
    print(
        f"{label:20} "
        f"P={metrics['p']:.4f} "
        f"R={metrics['r']:.4f} "
        f"F1={metrics['f']:.4f}"
    )

Precision: 1.0
Recall: 0.998
F1 Score: 0.999

Per-entity scores:
CASTE_CATEGORY       P=1.0000 R=1.0000 F1=1.0000
STATE                P=1.0000 R=1.0000 F1=1.0000
AGE                  P=1.0000 R=1.0000 F1=1.0000
OCCUPATION           P=1.0000 R=0.9883 F1=0.9941
ANNUAL_INCOME        P=1.0000 R=1.0000 F1=1.0000
DISABILITY           P=1.0000 R=1.0000 F1=1.0000
GENDER               P=1.0000 R=1.0000 F1=1.0000


In [37]:
unseen_tests = [
    "I am 32 years old and I work as a farmer in Bihar.",
    "My job is farming and I live in Rajasthan.",
    "I earn 150000 rupees every year and I am a farmer.",
    "Main kheti karta hoon aur Uttar Pradesh mein rehta hoon.",
    "Meri umar 28 saal hai aur main kisan hoon.",
    "I am a woman from Madhya Pradesh.",
    "Main ek divyang vyakti hoon.",
    "I belong to the OBC community and my income is 180000.",
    "I am 40 years old, unemployed, and from Maharashtra.",
    "Main 25 saal ka hoon aur mera occupation student hai."
]

for text in unseen_tests:
    test_ner_v3(text)


Input:
I am 32 years old and I work as a farmer in Bihar.

Extracted entities:
32                        → AGE
farmer                    → OCCUPATION
Bihar                     → STATE

Input:
My job is farming and I live in Rajasthan.

Extracted entities:
farming                   → GENDER
Rajasthan                 → STATE

Input:
I earn 150000 rupees every year and I am a farmer.

Extracted entities:
150000                    → ANNUAL_INCOME
farmer                    → OCCUPATION

Input:
Main kheti karta hoon aur Uttar Pradesh mein rehta hoon.

Extracted entities:
kheti                     → GENDER
karta                     → OCCUPATION
Uttar Pradesh             → STATE

Input:
Meri umar 28 saal hai aur main kisan hoon.

Extracted entities:
28                        → AGE
kisan                     → OCCUPATION

Input:
I am a woman from Madhya Pradesh.

Extracted entities:
Madhya Pradesh            → STATE

Input:
Main ek divyang vyakti hoon.

Extracted entities:
divyang              

In [38]:
final_augmentation_raw = [
    (
        "Main kheti karta hoon",
        [("kheti", "OCCUPATION")]
    ),
    (
        "Main kheti karta hoon aur Bihar mein rehta hoon",
        [
            ("kheti", "OCCUPATION"),
            ("Bihar", "STATE")
        ]
    ),
    (
        "I do farming for a living",
        [("farming", "OCCUPATION")]
    ),
    (
        "I am a woman",
        [("woman", "GENDER")]
    ),
    (
        "I am a woman from Uttar Pradesh",
        [
            ("woman", "GENDER"),
            ("Uttar Pradesh", "STATE")
        ]
    ),
    (
        "She is a woman from Maharashtra",
        [
            ("woman", "GENDER"),
            ("Maharashtra", "STATE")
        ]
    )
]


final_augmentation = []

for text, entities in final_augmentation_raw:
    spans = []

    for entity_text, label in entities:
        span = find_entity(text, entity_text, label)
        spans.append(span)

    final_augmentation.append(
        (text, {"entities": spans})
    )


print("Final augmentation examples:", len(final_augmentation))

for item in final_augmentation:
    print(item)

Final augmentation examples: 6
('Main kheti karta hoon', {'entities': [(5, 10, 'OCCUPATION')]})
('Main kheti karta hoon aur Bihar mein rehta hoon', {'entities': [(5, 10, 'OCCUPATION'), (26, 31, 'STATE')]})
('I do farming for a living', {'entities': [(5, 12, 'OCCUPATION')]})
('I am a woman', {'entities': [(7, 12, 'GENDER')]})
('I am a woman from Uttar Pradesh', {'entities': [(7, 12, 'GENDER'), (18, 31, 'STATE')]})
('She is a woman from Maharashtra', {'entities': [(9, 14, 'GENDER'), (20, 31, 'STATE')]})


In [39]:
from spacy.training import Example

valid_final_examples = []

for text, annotations in final_augmentation:
    doc = nlp_v3.make_doc(text)
    example = Example.from_dict(doc, annotations)
    valid_final_examples.append(example)

print("Total final augmentation examples:", len(final_augmentation))
print("Valid final augmentation examples:", len(valid_final_examples))

Total final augmentation examples: 6
Valid final augmentation examples: 6


In [40]:
final_train_data = (
    train_data
    + augmentation_data_fixed
    + farmer_augmentation_fixed
    + final_augmentation
)

print("Original training examples:", len(train_data))
print("Previous augmentation:", len(augmentation_data_fixed))
print("Farmer augmentation:", len(farmer_augmentation_fixed))
print("Final augmentation:", len(final_augmentation))
print("Total final training examples:", len(final_train_data))

Original training examples: 1600
Previous augmentation: 10
Farmer augmentation: 10
Final augmentation: 6
Total final training examples: 1626


In [41]:
import spacy

# Fresh model
nlp_final = spacy.blank("en")

# Add NER pipeline
ner_final = nlp_final.add_pipe("ner")

# Add all 7 entity labels
for label in ENTITY_LABELS:
    ner_final.add_label(label)

print("Pipeline:", nlp_final.pipe_names)
print("Entity labels:", ner_final.labels)

Pipeline: ['ner']
Entity labels: ('AGE', 'ANNUAL_INCOME', 'CASTE_CATEGORY', 'DISABILITY', 'GENDER', 'OCCUPATION', 'STATE')


In [42]:
import random
from spacy.training import Example
from spacy.util import minibatch, compounding

other_pipes = [pipe for pipe in nlp_final.pipe_names if pipe != "ner"]

with nlp_final.disable_pipes(*other_pipes):
    optimizer = nlp_final.begin_training()

    print("Starting FINAL Model 4 training...\n")

    for iteration in range(30):
        random.shuffle(final_train_data)
        losses = {}

        batches = minibatch(
            final_train_data,
            size=compounding(4.0, 32.0, 1.5)
        )

        for batch in batches:
            examples = []

            for text, annotations in batch:
                doc = nlp_final.make_doc(text)
                example = Example.from_dict(doc, annotations)
                examples.append(example)

            nlp_final.update(
                examples,
                drop=0.3,
                losses=losses
            )

        print(
            f"Iteration {iteration + 1}/30 "
            f"Loss: {losses.get('ner', 0):.4f}"
        )

print("\nFINAL Model 4 training completed!")

Starting FINAL Model 4 training...

Iteration 1/30 Loss: 9494.6562
Iteration 2/30 Loss: 388.3516
Iteration 3/30 Loss: 65.0956
Iteration 4/30 Loss: 47.5451
Iteration 5/30 Loss: 57.8006
Iteration 6/30 Loss: 59.6076
Iteration 7/30 Loss: 48.1710
Iteration 8/30 Loss: 56.3218
Iteration 9/30 Loss: 47.4608
Iteration 10/30 Loss: 47.1656
Iteration 11/30 Loss: 51.5162
Iteration 12/30 Loss: 50.9171
Iteration 13/30 Loss: 47.0582
Iteration 14/30 Loss: 53.0949
Iteration 15/30 Loss: 48.9757
Iteration 16/30 Loss: 49.9151
Iteration 17/30 Loss: 49.2052
Iteration 18/30 Loss: 55.5715
Iteration 19/30 Loss: 49.9280
Iteration 20/30 Loss: 50.6742
Iteration 21/30 Loss: 47.4273
Iteration 22/30 Loss: 54.0256
Iteration 23/30 Loss: 50.5436
Iteration 24/30 Loss: 46.0140
Iteration 25/30 Loss: 46.8869
Iteration 26/30 Loss: 47.7926
Iteration 27/30 Loss: 44.8662
Iteration 28/30 Loss: 45.7718
Iteration 29/30 Loss: 46.8506
Iteration 30/30 Loss: 46.4775

FINAL Model 4 training completed!


In [43]:
def test_final_ner(text):
    doc = nlp_final(text)

    print("\nInput:")
    print(text)

    print("\nExtracted entities:")
    if doc.ents:
        for ent in doc.ents:
            print(f"{ent.text:25} → {ent.label_}")
    else:
        print("No entities detected")


final_unseen_tests = [
    "I am 32 years old and I work as a farmer in Bihar.",
    "My job is farming and I live in Rajasthan.",
    "I earn 150000 rupees every year and I am a farmer.",
    "Main kheti karta hoon aur Uttar Pradesh mein rehta hoon.",
    "Meri umar 28 saal hai aur main kisan hoon.",
    "I am a woman from Madhya Pradesh.",
    "Main ek divyang vyakti hoon.",
    "I belong to the OBC community and my income is 180000.",
    "I am 40 years old, unemployed, and from Maharashtra.",
    "Main 25 saal ka hoon aur mera occupation student hai."
]

for text in final_unseen_tests:
    test_final_ner(text)


Input:
I am 32 years old and I work as a farmer in Bihar.

Extracted entities:
32                        → AGE
farmer                    → OCCUPATION
Bihar                     → STATE

Input:
My job is farming and I live in Rajasthan.

Extracted entities:
farming                   → OCCUPATION
Rajasthan                 → STATE

Input:
I earn 150000 rupees every year and I am a farmer.

Extracted entities:
150000                    → ANNUAL_INCOME
farmer                    → OCCUPATION

Input:
Main kheti karta hoon aur Uttar Pradesh mein rehta hoon.

Extracted entities:
kheti                     → OCCUPATION
Uttar Pradesh             → STATE

Input:
Meri umar 28 saal hai aur main kisan hoon.

Extracted entities:
28                        → AGE
kisan                     → OCCUPATION

Input:
I am a woman from Madhya Pradesh.

Extracted entities:
woman                     → GENDER
Madhya Pradesh            → STATE

Input:
Main ek divyang vyakti hoon.

Extracted entities:
divyang          

In [44]:
divyang_fix_raw = [
    (
        "Main divyang hoon",
        [("divyang", "DISABILITY")]
    ),
    (
        "Main ek divyang vyakti hoon",
        [("divyang", "DISABILITY")]
    ),
    (
        "I am a divyang person",
        [("divyang", "DISABILITY")]
    ),
    (
        "He is divyang",
        [("divyang", "DISABILITY")]
    ),
    (
        "She is divyang",
        [("divyang", "DISABILITY")]
    )
]

divyang_fix = []

for text, entities in divyang_fix_raw:
    spans = []

    for entity_text, label in entities:
        spans.append(
            find_entity(text, entity_text, label)
        )

    divyang_fix.append(
        (text, {"entities": spans})
    )

print("Divyang fix examples:", len(divyang_fix))

for item in divyang_fix:
    print(item)

Divyang fix examples: 5
('Main divyang hoon', {'entities': [(5, 12, 'DISABILITY')]})
('Main ek divyang vyakti hoon', {'entities': [(8, 15, 'DISABILITY')]})
('I am a divyang person', {'entities': [(7, 14, 'DISABILITY')]})
('He is divyang', {'entities': [(6, 13, 'DISABILITY')]})
('She is divyang', {'entities': [(7, 14, 'DISABILITY')]})


In [45]:
from spacy.training import Example

valid_divyang_examples = []

for text, annotations in divyang_fix:
    doc = nlp_final.make_doc(text)
    example = Example.from_dict(doc, annotations)
    valid_divyang_examples.append(example)

print("Total divyang fix examples:", len(divyang_fix))
print("Valid divyang fix examples:", len(valid_divyang_examples))

Total divyang fix examples: 5
Valid divyang fix examples: 5


In [46]:
final_train_data_v2 = (
    final_train_data
    + divyang_fix
)

print("Previous final training examples:", len(final_train_data))
print("Divyang fix examples:", len(divyang_fix))
print("Total corrected training examples:", len(final_train_data_v2))

Previous final training examples: 1626
Divyang fix examples: 5
Total corrected training examples: 1631


In [47]:
import random
from spacy.training import Example
from spacy.util import minibatch, compounding

# Create a completely fresh model
nlp_final_v2 = spacy.blank("en")
ner_final_v2 = nlp_final_v2.add_pipe("ner")

# Add all entity labels
for label in ENTITY_LABELS:
    ner_final_v2.add_label(label)

print("Pipeline:", nlp_final_v2.pipe_names)
print("Labels:", ner_final_v2.labels)

# Train
optimizer = nlp_final_v2.begin_training()

print("\nStarting corrected FINAL Model 4 training...\n")

for iteration in range(30):
    random.shuffle(final_train_data_v2)
    losses = {}

    batches = minibatch(
        final_train_data_v2,
        size=compounding(4.0, 32.0, 1.5)
    )

    for batch in batches:
        examples = []

        for text, annotations in batch:
            doc = nlp_final_v2.make_doc(text)
            example = Example.from_dict(doc, annotations)
            examples.append(example)

        nlp_final_v2.update(
            examples,
            drop=0.3,
            losses=losses
        )

    print(
        f"Iteration {iteration + 1}/30 "
        f"Loss: {losses.get('ner', 0):.4f}"
    )

print("\nCorrected FINAL Model 4 training completed!")

Pipeline: ['ner']
Labels: ('AGE', 'ANNUAL_INCOME', 'CASTE_CATEGORY', 'DISABILITY', 'GENDER', 'OCCUPATION', 'STATE')

Starting corrected FINAL Model 4 training...

Iteration 1/30 Loss: 9363.5859
Iteration 2/30 Loss: 435.8788
Iteration 3/30 Loss: 72.7185
Iteration 4/30 Loss: 59.4025
Iteration 5/30 Loss: 55.1990
Iteration 6/30 Loss: 57.0231
Iteration 7/30 Loss: 47.0989
Iteration 8/30 Loss: 49.0844
Iteration 9/30 Loss: 52.4791
Iteration 10/30 Loss: 50.2446
Iteration 11/30 Loss: 45.8576
Iteration 12/30 Loss: 46.1000
Iteration 13/30 Loss: 45.8767
Iteration 14/30 Loss: 52.9137
Iteration 15/30 Loss: 45.4421
Iteration 16/30 Loss: 56.2560
Iteration 17/30 Loss: 50.2675
Iteration 18/30 Loss: 51.5274
Iteration 19/30 Loss: 50.5472
Iteration 20/30 Loss: 48.7863
Iteration 21/30 Loss: 47.0657
Iteration 22/30 Loss: 46.4650
Iteration 23/30 Loss: 45.2554
Iteration 24/30 Loss: 66.9007
Iteration 25/30 Loss: 47.5997
Iteration 26/30 Loss: 52.3866
Iteration 27/30 Loss: 49.5557
Iteration 28/30 Loss: 46.9784
Ite

In [48]:
def test_ner_final(text):
    doc = nlp_final_v2(text)

    print("\nInput:")
    print(text)

    print("\nExtracted entities:")

    if doc.ents:
        for ent in doc.ents:
            print(f"{ent.text:25} → {ent.label_}")
    else:
        print("No entities detected")


final_unseen_tests = [
    "I am 32 years old and I work as a farmer in Bihar.",
    "My job is farming and I live in Rajasthan.",
    "I earn 150000 rupees every year and I am a farmer.",
    "Main kheti karta hoon aur Uttar Pradesh mein rehta hoon.",
    "Meri umar 28 saal hai aur main kisan hoon.",
    "I am a woman from Madhya Pradesh.",
    "Main ek divyang vyakti hoon.",
    "I belong to the OBC community and my income is 180000.",
    "I am 40 years old, unemployed, and from Maharashtra.",
    "Main 25 saal ka hoon aur mera occupation student hai."
]

for text in final_unseen_tests:
    test_ner_final(text)


Input:
I am 32 years old and I work as a farmer in Bihar.

Extracted entities:
32                        → AGE
farmer                    → OCCUPATION
Bihar                     → STATE

Input:
My job is farming and I live in Rajasthan.

Extracted entities:
farming                   → OCCUPATION
Rajasthan                 → STATE

Input:
I earn 150000 rupees every year and I am a farmer.

Extracted entities:
150000                    → ANNUAL_INCOME
farmer                    → OCCUPATION

Input:
Main kheti karta hoon aur Uttar Pradesh mein rehta hoon.

Extracted entities:
kheti                     → OCCUPATION
Uttar Pradesh             → STATE
.                         → GENDER

Input:
Meri umar 28 saal hai aur main kisan hoon.

Extracted entities:
28                        → AGE
kisan                     → OCCUPATION

Input:
I am a woman from Madhya Pradesh.

Extracted entities:
woman                     → GENDER
Madhya Pradesh            → STATE

Input:
Main ek divyang vyakti hoon.

Ex

In [49]:
# Add punctuation-only examples so the model learns
# that punctuation should NOT be an entity.

punctuation_fix = [
    ("Main kheti karta hoon.", {
        "entities": [(5, 10, "OCCUPATION")]
    }),
    
    ("Main ek divyang vyakti hoon.", {
        "entities": [(8, 15, "DISABILITY")]
    }),
    
    ("Main 25 saal ka hoon.", {
        "entities": [(5, 7, "AGE")]
    }),
    
    ("I am a woman.", {
        "entities": [(7, 12, "GENDER")]
    }),
    
    ("I am a farmer.", {
        "entities": [(7, 13, "OCCUPATION")]
    }),
    
    ("I live in Bihar.", {
        "entities": [(10, 15, "STATE")]
    })
]

print("Punctuation-fix examples:", len(punctuation_fix))

Punctuation-fix examples: 6


In [50]:
# Combine the original final training data with punctuation fixes

final_train_data_v3 = final_train_data_v2 + punctuation_fix

print("Previous training examples:", len(final_train_data_v2))
print("Punctuation fix examples:", len(punctuation_fix))
print("Total final training examples:", len(final_train_data_v3))

Previous training examples: 1631
Punctuation fix examples: 6
Total final training examples: 1637


In [51]:
# Combine the original final training data with punctuation fixes

final_train_data_v3 = final_train_data_v2 + punctuation_fix

print("Previous training examples:", len(final_train_data_v2))
print("Punctuation fix examples:", len(punctuation_fix))
print("Total final training examples:", len(final_train_data_v3))

Previous training examples: 1631
Punctuation fix examples: 6
Total final training examples: 1637


In [52]:
import spacy
import random
from spacy.training import Example
from spacy.util import minibatch, compounding

# Create a completely fresh NER model
nlp_final_v3 = spacy.blank("en")
ner_final_v3 = nlp_final_v3.add_pipe("ner")

# Add entity labels
ENTITY_LABELS = [
    "AGE",
    "GENDER",
    "OCCUPATION",
    "STATE",
    "ANNUAL_INCOME",
    "CASTE_CATEGORY",
    "DISABILITY"
]

for label in ENTITY_LABELS:
    ner_final_v3.add_label(label)

print("Pipeline:", nlp_final_v3.pipe_names)
print("Labels:", ner_final_v3.labels)

# Initialize training
optimizer = nlp_final_v3.begin_training()

print("\nStarting FINAL Model 4 v3 training...\n")

# Training loop
for iteration in range(30):

    random.shuffle(final_train_data_v3)
    losses = {}

    batches = minibatch(
        final_train_data_v3,
        size=compounding(4.0, 32.0, 1.5)
    )

    for batch in batches:

        examples = []

        for text, annotations in batch:

            doc = nlp_final_v3.make_doc(text)

            example = Example.from_dict(
                doc,
                annotations
            )

            examples.append(example)

        nlp_final_v3.update(
            examples,
            drop=0.3,
            losses=losses
        )

    print(
        f"Iteration {iteration + 1}/30 "
        f"Loss: {losses.get('ner', 0):.4f}"
    )

print("\nFINAL Model 4 v3 training completed!")

Pipeline: ['ner']
Labels: ('AGE', 'ANNUAL_INCOME', 'CASTE_CATEGORY', 'DISABILITY', 'GENDER', 'OCCUPATION', 'STATE')

Starting FINAL Model 4 v3 training...

Iteration 1/30 Loss: 9617.0293
Iteration 2/30 Loss: 341.8047
Iteration 3/30 Loss: 66.0778
Iteration 4/30 Loss: 55.1451
Iteration 5/30 Loss: 49.8321
Iteration 6/30 Loss: 51.6310
Iteration 7/30 Loss: 52.0312
Iteration 8/30 Loss: 47.6086
Iteration 9/30 Loss: 47.5942
Iteration 10/30 Loss: 50.2305
Iteration 11/30 Loss: 52.4661
Iteration 12/30 Loss: 45.6463
Iteration 13/30 Loss: 45.5264
Iteration 14/30 Loss: 57.8121
Iteration 15/30 Loss: 43.4659
Iteration 16/30 Loss: 48.0433
Iteration 17/30 Loss: 50.8278
Iteration 18/30 Loss: 44.4319
Iteration 19/30 Loss: 46.9666
Iteration 20/30 Loss: 45.8902
Iteration 21/30 Loss: 45.4459
Iteration 22/30 Loss: 46.3995
Iteration 23/30 Loss: 42.9197
Iteration 24/30 Loss: 47.2640
Iteration 25/30 Loss: 47.1749
Iteration 26/30 Loss: 45.2429
Iteration 27/30 Loss: 53.1376
Iteration 28/30 Loss: 59.0242
Iteration 

In [53]:
def test_ner_v3(text):
    doc = nlp_final_v3(text)

    print("\nInput:")
    print(text)

    print("\nExtracted entities:")

    if doc.ents:
        for ent in doc.ents:
            print(f"{ent.text:25} → {ent.label_}")
    else:
        print("No entities detected")


final_unseen_tests = [
    "I am 32 years old and I work as a farmer in Bihar.",
    "My job is farming and I live in Rajasthan.",
    "I earn 150000 rupees every year and I am a farmer.",
    "Main kheti karta hoon aur Uttar Pradesh mein rehta hoon.",
    "Meri umar 28 saal hai aur main kisan hoon.",
    "I am a woman from Madhya Pradesh.",
    "Main ek divyang vyakti hoon.",
    "I belong to the OBC community and my income is 180000.",
    "I am 40 years old, unemployed, and from Maharashtra.",
    "Main 25 saal ka hoon aur mera occupation student hai."
]

for text in final_unseen_tests:
    test_ner_v3(text)


Input:
I am 32 years old and I work as a farmer in Bihar.

Extracted entities:
32                        → AGE
farmer                    → OCCUPATION
Bihar                     → STATE

Input:
My job is farming and I live in Rajasthan.

Extracted entities:
farming                   → OCCUPATION
Rajasthan                 → STATE

Input:
I earn 150000 rupees every year and I am a farmer.

Extracted entities:
150000                    → ANNUAL_INCOME
farmer                    → OCCUPATION

Input:
Main kheti karta hoon aur Uttar Pradesh mein rehta hoon.

Extracted entities:
kheti                     → OCCUPATION
Uttar Pradesh             → STATE

Input:
Meri umar 28 saal hai aur main kisan hoon.

Extracted entities:
28                        → AGE
kisan                     → OCCUPATION

Input:
I am a woman from Madhya Pradesh.

Extracted entities:
woman                     → GENDER
Madhya Pradesh            → STATE

Input:
Main ek divyang vyakti hoon.

Extracted entities:
divyang          

In [54]:
student_fix_raw = [
    (
        "I am a student",
        [("student", "OCCUPATION")]
    ),
    (
        "I am a student from Uttar Pradesh",
        [
            ("student", "OCCUPATION"),
            ("Uttar Pradesh", "STATE")
        ]
    ),
    (
        "My occupation is student",
        [("student", "OCCUPATION")]
    ),
    (
        "Main student hoon",
        [("student", "OCCUPATION")]
    ),
    (
        "Main ek student hoon",
        [("student", "OCCUPATION")]
    ),
    (
        "Main 25 saal ka hoon aur mera occupation student hai",
        [
            ("25", "AGE"),
            ("student", "OCCUPATION")
        ]
    )
]

student_fix = []

for text, entities in student_fix_raw:
    spans = []

    for entity_text, label in entities:
        spans.append(
            find_entity(text, entity_text, label)
        )

    student_fix.append(
        (text, {"entities": spans})
    )

print("Student-fix examples:", len(student_fix))

Student-fix examples: 6


In [55]:
# Add student fixes to the final training data

final_train_data_v4 = final_train_data_v3 + student_fix

print("Previous training examples:", len(final_train_data_v3))
print("Student-fix examples:", len(student_fix))
print("Total FINAL training examples:", len(final_train_data_v4))

Previous training examples: 1637
Student-fix examples: 6
Total FINAL training examples: 1643


In [56]:
import spacy
import random
from spacy.training import Example
from spacy.util import minibatch, compounding

# Entity labels
ENTITY_LABELS = [
    "AGE",
    "GENDER",
    "OCCUPATION",
    "STATE",
    "ANNUAL_INCOME",
    "CASTE_CATEGORY",
    "DISABILITY"
]

# Create a completely fresh model
nlp_model4 = spacy.blank("en")
ner_model4 = nlp_model4.add_pipe("ner")

# Add labels
for label in ENTITY_LABELS:
    ner_model4.add_label(label)

print("Pipeline:", nlp_model4.pipe_names)
print("Labels:", ner_model4.labels)

# Initialize training
optimizer = nlp_model4.begin_training()

print("\nStarting FINAL Model 4 training...\n")

for iteration in range(30):

    random.shuffle(final_train_data_v4)
    losses = {}

    batches = minibatch(
        final_train_data_v4,
        size=compounding(4.0, 32.0, 1.5)
    )

    for batch in batches:

        examples = []

        for text, annotations in batch:

            doc = nlp_model4.make_doc(text)

            example = Example.from_dict(
                doc,
                annotations
            )

            examples.append(example)

        nlp_model4.update(
            examples,
            drop=0.3,
            losses=losses
        )

    print(
        f"Iteration {iteration + 1}/30 "
        f"Loss: {losses.get('ner', 0):.4f}"
    )

print("\nFINAL Model 4 training completed successfully!")

Pipeline: ['ner']
Labels: ('AGE', 'ANNUAL_INCOME', 'CASTE_CATEGORY', 'DISABILITY', 'GENDER', 'OCCUPATION', 'STATE')

Starting FINAL Model 4 training...

Iteration 1/30 Loss: 9600.8486
Iteration 2/30 Loss: 281.6776
Iteration 3/30 Loss: 75.3223
Iteration 4/30 Loss: 53.1067
Iteration 5/30 Loss: 48.0577
Iteration 6/30 Loss: 48.3119
Iteration 7/30 Loss: 48.1428
Iteration 8/30 Loss: 56.8969
Iteration 9/30 Loss: 50.9569
Iteration 10/30 Loss: 51.7441
Iteration 11/30 Loss: 52.9940
Iteration 12/30 Loss: 44.3073
Iteration 13/30 Loss: 48.4080
Iteration 14/30 Loss: 46.3532
Iteration 15/30 Loss: 45.2529
Iteration 16/30 Loss: 45.6754
Iteration 17/30 Loss: 46.5571
Iteration 18/30 Loss: 45.2442
Iteration 19/30 Loss: 53.1878
Iteration 20/30 Loss: 45.5562
Iteration 21/30 Loss: 45.8395
Iteration 22/30 Loss: 45.8103
Iteration 23/30 Loss: 57.1871
Iteration 24/30 Loss: 44.6575
Iteration 25/30 Loss: 43.2292
Iteration 26/30 Loss: 47.2080
Iteration 27/30 Loss: 49.7893
Iteration 28/30 Loss: 46.4508
Iteration 29/

In [57]:
def test_model4(text):
    doc = nlp_model4(text)

    print("\nInput:")
    print(text)

    print("\nExtracted entities:")

    if doc.ents:
        for ent in doc.ents:
            print(f"{ent.text:25} → {ent.label_}")
    else:
        print("No entities detected")


final_tests = [
    "I am 32 years old and I work as a farmer in Bihar.",
    "My job is farming and I live in Rajasthan.",
    "I earn 150000 rupees every year and I am a farmer.",
    "Main kheti karta hoon aur Uttar Pradesh mein rehta hoon.",
    "Meri umar 28 saal hai aur main kisan hoon.",
    "I am a woman from Madhya Pradesh.",
    "Main ek divyang vyakti hoon.",
    "I belong to the OBC community and my income is 180000.",
    "I am 40 years old, unemployed, and from Maharashtra.",
    "Main 25 saal ka hoon aur mera occupation student hai."
]

for text in final_tests:
    test_model4(text)


Input:
I am 32 years old and I work as a farmer in Bihar.

Extracted entities:
32                        → AGE
farmer                    → OCCUPATION
Bihar                     → STATE

Input:
My job is farming and I live in Rajasthan.

Extracted entities:
farming                   → OCCUPATION
Rajasthan                 → STATE

Input:
I earn 150000 rupees every year and I am a farmer.

Extracted entities:
150000                    → ANNUAL_INCOME
every                     → GENDER
farmer                    → OCCUPATION

Input:
Main kheti karta hoon aur Uttar Pradesh mein rehta hoon.

Extracted entities:
kheti                     → OCCUPATION
Uttar Pradesh             → STATE

Input:
Meri umar 28 saal hai aur main kisan hoon.

Extracted entities:
28                        → AGE
kisan                     → OCCUPATION

Input:
I am a woman from Madhya Pradesh.

Extracted entities:
woman                     → GENDER
Madhya Pradesh            → STATE

Input:
Main ek divyang vyakti hoon.

Ex

In [58]:
negative_fix = [
    ("I earn 150000 rupees every year", {
        "entities": [(7, 13, "ANNUAL_INCOME")]
    }),
    
    ("I am 40 years old unemployed", {
        "entities": [
            (5, 7, "AGE"),
            (19, 29, "OCCUPATION")
        ]
    }),
    
    ("I am 40 years old, unemployed, and from Maharashtra", {
        "entities": [
            (5, 7, "AGE"),
            (20, 30, "OCCUPATION"),
            (44, 55, "STATE")
        ]
    })
]

print("Negative/context-fix examples:", len(negative_fix))

Negative/context-fix examples: 3


In [59]:
# Add negative/context fixes

final_train_data_v5 = final_train_data_v4 + negative_fix

print("Previous training examples:", len(final_train_data_v4))
print("Negative/context-fix examples:", len(negative_fix))
print("Total FINAL training examples:", len(final_train_data_v5))

Previous training examples: 1643
Negative/context-fix examples: 3
Total FINAL training examples: 1646


In [60]:
import spacy
import random
from spacy.training import Example
from spacy.util import minibatch, compounding

# Create a completely fresh model
nlp_model4_final = spacy.blank("en")
ner_model4_final = nlp_model4_final.add_pipe("ner")

# Entity labels
ENTITY_LABELS = [
    "AGE",
    "GENDER",
    "OCCUPATION",
    "STATE",
    "ANNUAL_INCOME",
    "CASTE_CATEGORY",
    "DISABILITY"
]

# Add labels
for label in ENTITY_LABELS:
    ner_model4_final.add_label(label)

print("Pipeline:", nlp_model4_final.pipe_names)
print("Labels:", ner_model4_final.labels)

# Initialize training
optimizer = nlp_model4_final.begin_training()

print("\nStarting FINAL Model 4 training...\n")

for iteration in range(30):

    random.shuffle(final_train_data_v5)
    losses = {}

    batches = minibatch(
        final_train_data_v5,
        size=compounding(4.0, 32.0, 1.5)
    )

    for batch in batches:

        examples = []

        for text, annotations in batch:

            doc = nlp_model4_final.make_doc(text)

            example = Example.from_dict(
                doc,
                annotations
            )

            examples.append(example)

        nlp_model4_final.update(
            examples,
            drop=0.3,
            losses=losses
        )

    print(
        f"Iteration {iteration + 1}/30 "
        f"Loss: {losses.get('ner', 0):.4f}"
    )

print("\nFINAL Model 4 training completed successfully!")

Pipeline: ['ner']
Labels: ('AGE', 'ANNUAL_INCOME', 'CASTE_CATEGORY', 'DISABILITY', 'GENDER', 'OCCUPATION', 'STATE')

Starting FINAL Model 4 training...



c:\Users\hp\OneDrive\Desktop\Entity Extraction\venv\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "I am 40 years old, unemployed, and from Maharashtr..." with entities "[(5, 7, 'AGE'), (20, 30, 'OCCUPATION'), (44, 55, '...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
c:\Users\hp\OneDrive\Desktop\Entity Extraction\venv\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "I am 40 years old unemployed" with entities "[(5, 7, 'AGE'), (19, 29, 'OCCUPATION')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(


Iteration 1/30 Loss: 9587.6582
Iteration 2/30 Loss: 402.7582
Iteration 3/30 Loss: 74.8946
Iteration 4/30 Loss: 50.6950
Iteration 5/30 Loss: 49.3372
Iteration 6/30 Loss: 50.7239
Iteration 7/30 Loss: 58.8790
Iteration 8/30 Loss: 49.6034
Iteration 9/30 Loss: 45.7441
Iteration 10/30 Loss: 45.0851
Iteration 11/30 Loss: 49.6407
Iteration 12/30 Loss: 45.4120
Iteration 13/30 Loss: 46.9227
Iteration 14/30 Loss: 48.9248
Iteration 15/30 Loss: 45.4637
Iteration 16/30 Loss: 53.4992
Iteration 17/30 Loss: 44.7508
Iteration 18/30 Loss: 49.6842
Iteration 19/30 Loss: 44.2452
Iteration 20/30 Loss: 52.7426
Iteration 21/30 Loss: 44.4922
Iteration 22/30 Loss: 57.2010
Iteration 23/30 Loss: 50.7153
Iteration 24/30 Loss: 47.7975
Iteration 25/30 Loss: 52.3520
Iteration 26/30 Loss: 48.9644
Iteration 27/30 Loss: 45.8582
Iteration 28/30 Loss: 46.9361
Iteration 29/30 Loss: 48.7640
Iteration 30/30 Loss: 52.1065

FINAL Model 4 training completed successfully!


In [61]:
def test_model4_final(text):
    doc = nlp_model4_final(text)

    print("\nInput:")
    print(text)

    print("\nExtracted entities:")

    if doc.ents:
        for ent in doc.ents:
            print(f"{ent.text:25} → {ent.label_}")
    else:
        print("No entities detected")


final_validation_tests = [
    "I am 32 years old and I work as a farmer in Bihar.",
    "My job is farming and I live in Rajasthan.",
    "I earn 150000 rupees every year and I am a farmer.",
    "Main kheti karta hoon aur Uttar Pradesh mein rehta hoon.",
    "Meri umar 28 saal hai aur main kisan hoon.",
    "I am a woman from Madhya Pradesh.",
    "Main ek divyang vyakti hoon.",
    "I belong to the OBC community and my income is 180000.",
    "I am 40 years old, unemployed, and from Maharashtra.",
    "Main 25 saal ka hoon aur mera occupation student hai."
]

for text in final_validation_tests:
    test_model4_final(text)


Input:
I am 32 years old and I work as a farmer in Bihar.

Extracted entities:
32                        → AGE
farmer                    → OCCUPATION
Bihar                     → STATE

Input:
My job is farming and I live in Rajasthan.

Extracted entities:
farming                   → OCCUPATION
Rajasthan                 → STATE

Input:
I earn 150000 rupees every year and I am a farmer.

Extracted entities:
150000                    → ANNUAL_INCOME
farmer                    → OCCUPATION

Input:
Main kheti karta hoon aur Uttar Pradesh mein rehta hoon.

Extracted entities:
kheti                     → OCCUPATION
Uttar Pradesh             → STATE

Input:
Meri umar 28 saal hai aur main kisan hoon.

Extracted entities:
28                        → AGE
kisan                     → OCCUPATION

Input:
I am a woman from Madhya Pradesh.

Extracted entities:
woman                     → GENDER
Madhya Pradesh            → STATE

Input:
Main ek divyang vyakti hoon.

Extracted entities:
divyang          

In [62]:
obc_fix = [
    (
        "I belong to the OBC community",
        {
            "entities": [
                (17, 20, "CASTE_CATEGORY")
            ]
        }
    ),
    (
        "I belong to OBC category",
        {
            "entities": [
                (13, 16, "CASTE_CATEGORY")
            ]
        }
    ),
    (
        "My caste category is OBC",
        {
            "entities": [
                (22, 25, "CASTE_CATEGORY")
            ]
        }
    )
]

print("OBC context-fix examples:", len(obc_fix))

OBC context-fix examples: 3


In [63]:
# Add OBC context fixes

final_train_data_v6 = final_train_data_v5 + obc_fix

print("Previous training examples:", len(final_train_data_v5))
print("OBC context-fix examples:", len(obc_fix))
print("Total FINAL training examples:", len(final_train_data_v6))

Previous training examples: 1646
OBC context-fix examples: 3
Total FINAL training examples: 1649


In [65]:
import spacy
import random
from spacy.training import Example
from spacy.util import minibatch, compounding

# Create fresh final model
nlp_model4_final_v2 = spacy.blank("en")
ner_model4_final_v2 = nlp_model4_final_v2.add_pipe("ner")

# Entity labels
ENTITY_LABELS = [
    "AGE",
    "GENDER",
    "OCCUPATION",
    "STATE",
    "ANNUAL_INCOME",
    "CASTE_CATEGORY",
    "DISABILITY"
]

# Add labels
for label in ENTITY_LABELS:
    ner_model4_final_v2.add_label(label)

print("Pipeline:", nlp_model4_final_v2.pipe_names)
print("Labels:", ner_model4_final_v2.labels)

# Initialize training
optimizer = nlp_model4_final_v2.begin_training()

print("\nStarting FINAL Model 4 training...\n")

for iteration in range(30):

    random.shuffle(final_train_data_v6)
    losses = {}

    batches = minibatch(
        final_train_data_v6,
        size=compounding(4.0, 32.0, 1.5)
    )

    for batch in batches:

        examples = []

        for text, annotations in batch:

            doc = nlp_model4_final_v2.make_doc(text)

            example = Example.from_dict(
                doc,
                annotations
            )

            examples.append(example)

        nlp_model4_final_v2.update(
            examples,
            drop=0.3,
            losses=losses
        )

    print(
        f"Iteration {iteration + 1}/30 "
        f"Loss: {losses.get('ner', 0):.4f}"
    )

print("\nFINAL Model 4 training completed successfully!")

Pipeline: ['ner']
Labels: ('AGE', 'ANNUAL_INCOME', 'CASTE_CATEGORY', 'DISABILITY', 'GENDER', 'OCCUPATION', 'STATE')

Starting FINAL Model 4 training...

Iteration 1/30 Loss: 9763.4844
Iteration 2/30 Loss: 385.0865
Iteration 3/30 Loss: 67.6476
Iteration 4/30 Loss: 99.2863
Iteration 5/30 Loss: 49.5516
Iteration 6/30 Loss: 51.2041
Iteration 7/30 Loss: 56.1677
Iteration 8/30 Loss: 51.7938
Iteration 9/30 Loss: 50.7898
Iteration 10/30 Loss: 47.2023
Iteration 11/30 Loss: 55.1081
Iteration 12/30 Loss: 57.8565
Iteration 13/30 Loss: 45.7645
Iteration 14/30 Loss: 47.9954
Iteration 15/30 Loss: 49.9764
Iteration 16/30 Loss: 50.8794
Iteration 17/30 Loss: 50.3889
Iteration 18/30 Loss: 45.9012
Iteration 19/30 Loss: 46.3526
Iteration 20/30 Loss: 52.0265
Iteration 21/30 Loss: 45.6802
Iteration 22/30 Loss: 45.5435
Iteration 23/30 Loss: 54.6779
Iteration 24/30 Loss: 48.2108
Iteration 25/30 Loss: 52.1072
Iteration 26/30 Loss: 45.7998
Iteration 27/30 Loss: 49.2388
Iteration 28/30 Loss: 48.6569
Iteration 29/

In [66]:
def test_model4_final_v2(text):
    doc = nlp_model4_final_v2(text)

    print("\nInput:")
    print(text)

    print("\nExtracted entities:")

    if doc.ents:
        for ent in doc.ents:
            print(f"{ent.text:25} → {ent.label_}")
    else:
        print("No entities detected")


final_validation_tests = [
    "I am 32 years old and I work as a farmer in Bihar.",
    "My job is farming and I live in Rajasthan.",
    "I earn 150000 rupees every year and I am a farmer.",
    "Main kheti karta hoon aur Uttar Pradesh mein rehta hoon.",
    "Meri umar 28 saal hai aur main kisan hoon.",
    "I am a woman from Madhya Pradesh.",
    "Main ek divyang vyakti hoon.",
    "I belong to the OBC community and my income is 180000.",
    "I am 40 years old, unemployed, and from Maharashtra.",
    "Main 25 saal ka hoon aur mera occupation student hai."
]

for text in final_validation_tests:
    test_model4_final_v2(text)


Input:
I am 32 years old and I work as a farmer in Bihar.

Extracted entities:
32                        → AGE
farmer                    → OCCUPATION
Bihar                     → STATE

Input:
My job is farming and I live in Rajasthan.

Extracted entities:
farming                   → OCCUPATION
Rajasthan                 → STATE

Input:
I earn 150000 rupees every year and I am a farmer.

Extracted entities:
150000                    → ANNUAL_INCOME
farmer                    → OCCUPATION

Input:
Main kheti karta hoon aur Uttar Pradesh mein rehta hoon.

Extracted entities:
kheti                     → OCCUPATION
Uttar Pradesh             → STATE

Input:
Meri umar 28 saal hai aur main kisan hoon.

Extracted entities:
28                        → AGE
kisan                     → OCCUPATION

Input:
I am a woman from Madhya Pradesh.

Extracted entities:
woman                     → GENDER
Madhya Pradesh            → STATE

Input:
Main ek divyang vyakti hoon.

Extracted entities:
divyang          

In [69]:
from spacy.training import Example
from spacy.scorer import Scorer

examples = []

for text, annotations in test_data:
    pred_doc = nlp_model4_final_v2(text)
    example = Example.from_dict(pred_doc, annotations)
    examples.append(example)

scorer = Scorer()
scores = scorer.score(examples)

print("Overall Performance")
print("-------------------")
print("Precision:", round(scores["ents_p"], 4))
print("Recall:   ", round(scores["ents_r"], 4))
print("F1 Score: ", round(scores["ents_f"], 4))

print("\nPer-entity scores")
print("-----------------")

for label, metrics in scores["ents_per_type"].items():
    precision = metrics.get("p", 0)
    recall = metrics.get("r", 0)

    if precision + recall > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0

    print(
        f"{label:20} "
        f"P={precision:.4f} "
        f"R={recall:.4f} "
        f"F1={f1:.4f}"
    )

Overall Performance
-------------------
Precision: 1.0
Recall:    0.998
F1 Score:  0.999

Per-entity scores
-----------------
CASTE_CATEGORY       P=1.0000 R=1.0000 F1=1.0000
STATE                P=1.0000 R=1.0000 F1=1.0000
AGE                  P=1.0000 R=1.0000 F1=1.0000
OCCUPATION           P=1.0000 R=0.9883 F1=0.9941
ANNUAL_INCOME        P=1.0000 R=1.0000 F1=1.0000
DISABILITY           P=1.0000 R=1.0000 F1=1.0000
GENDER               P=1.0000 R=1.0000 F1=1.0000


In [70]:
from pathlib import Path

model_path = Path("../models/model4_ner")
model_path.parent.mkdir(parents=True, exist_ok=True)

nlp_model4_final_v2.to_disk(model_path)

print("Model 4 saved successfully!")
print("Saved to:", model_path.resolve())

Model 4 saved successfully!
Saved to: C:\Users\hp\OneDrive\Desktop\Entity Extraction\models\model4_ner


In [71]:
import spacy

loaded_model = spacy.load("../models/model4_ner")

print("Model 4 loaded successfully!")
print("Pipeline:", loaded_model.pipe_names)
print("NER labels:", loaded_model.get_pipe("ner").labels)

Model 4 loaded successfully!
Pipeline: ['ner']
NER labels: ('AGE', 'ANNUAL_INCOME', 'CASTE_CATEGORY', 'DISABILITY', 'GENDER', 'OCCUPATION', 'STATE')


In [72]:
def extract_user_profile(text):
    doc = loaded_model(text)

    profile = {
        "age": None,
        "gender": None,
        "occupation": None,
        "state": None,
        "annual_income": None,
        "caste_category": None,
        "disability": None
    }

    for ent in doc.ents:
        key = ent.label_.lower()

        if key in profile and profile[key] is None:
            profile[key] = ent.text

    return profile

In [73]:
text = "I am a 25 year old student from Uttar Pradesh and my annual income is 120000."

profile = extract_user_profile(text)

print("User Input:")
print(text)

print("\nExtracted Profile:")
for key, value in profile.items():
    print(f"{key:20} : {value}")

User Input:
I am a 25 year old student from Uttar Pradesh and my annual income is 120000.

Extracted Profile:
age                  : 25
gender               : None
occupation           : None
state                : Uttar Pradesh
annual_income        : 120000
caste_category       : None
disability           : None


In [77]:
student_context_fix = [
    (
        "I am a 25 year old student from Uttar Pradesh",
        [
            ("25", "AGE"),
            ("student", "OCCUPATION"),
            ("Uttar Pradesh", "STATE")
        ]
    ),
    (
        "I am a student from Uttar Pradesh and my annual income is 120000",
        [
            ("student", "OCCUPATION"),
            ("Uttar Pradesh", "STATE"),
            ("120000", "ANNUAL_INCOME")
        ]
    ),
    (
        "I am a 25 year old student",
        [
            ("25", "AGE"),
            ("student", "OCCUPATION")
        ]
    ),
    (
        "My age is 25 and I am a student from Uttar Pradesh",
        [
            ("25", "AGE"),
            ("student", "OCCUPATION"),
            ("Uttar Pradesh", "STATE")
        ]
    )
]

# Automatically calculate correct character offsets
student_context_fix_spacy = []

for text, entities in student_context_fix:
    annotations = []

    for entity_text, label in entities:
        start = text.find(entity_text)

        if start == -1:
            print("ERROR:", entity_text, "not found in:", text)
        else:
            end = start + len(entity_text)
            annotations.append((start, end, label))
            print(f"{entity_text!r} → ({start}, {end}) → {label}")

    student_context_fix_spacy.append(
        (text, {"entities": annotations})
    )

print("\nAll offsets calculated automatically.")

'25' → (7, 9) → AGE
'student' → (19, 26) → OCCUPATION
'Uttar Pradesh' → (32, 45) → STATE
'student' → (7, 14) → OCCUPATION
'Uttar Pradesh' → (20, 33) → STATE
'120000' → (58, 64) → ANNUAL_INCOME
'25' → (7, 9) → AGE
'student' → (19, 26) → OCCUPATION
'25' → (10, 12) → AGE
'student' → (24, 31) → OCCUPATION
'Uttar Pradesh' → (37, 50) → STATE

All offsets calculated automatically.


In [78]:
from spacy.training import Example

validated_student_examples = []

for text, annotations in student_context_fix_spacy:
    doc = loaded_model.make_doc(text)
    example = Example.from_dict(doc, annotations)
    validated_student_examples.append(example)

print("Total student examples:", len(validated_student_examples))
print("All student examples validated successfully!")

Total student examples: 4
All student examples validated successfully!


In [79]:
# Add the new student-context examples
train_data_v2 = train_data + [
    (text, annotations)
    for text, annotations in student_context_fix_spacy
]

print("Original training examples:", len(train_data))
print("New student examples:", len(student_context_fix_spacy))
print("Updated training examples:", len(train_data_v2))

Original training examples: 1600
New student examples: 4
Updated training examples: 1604


In [81]:
print("Available variables containing 'train':")

for name in list(globals().keys()):
    if "train" in name.lower():
        value = globals()[name]
        try:
            print(name, "→", len(value))
        except:
            pass

Available variables containing 'train':
training_data → 2000
train_data → 1600
improved_train_data → 1610
improved_train_data_v3 → 1620
final_train_data → 1626
final_train_data_v2 → 1631
final_train_data_v3 → 1637
final_train_data_v4 → 1643
final_train_data_v5 → 1646
final_train_data_v6 → 1649
train_data_v2 → 1604


In [82]:
final_train_data_v7 = final_train_data_v6 + [
    (text, annotations)
    for text, annotations in student_context_fix_spacy
]

print("Previous training examples:", len(final_train_data_v6))
print("New student examples:", len(student_context_fix_spacy))
print("Final training examples:", len(final_train_data_v7))

Previous training examples: 1649
New student examples: 4
Final training examples: 1653


In [83]:
import spacy
import random
from spacy.training import Example
from spacy.util import minibatch, compounding

# Create a fresh blank English model
nlp_model4_v2 = spacy.blank("en")

# Add NER pipeline
ner = nlp_model4_v2.add_pipe("ner")

# Add all entity labels
ENTITY_LABELS = [
    "AGE",
    "GENDER",
    "OCCUPATION",
    "STATE",
    "ANNUAL_INCOME",
    "CASTE_CATEGORY",
    "DISABILITY"
]

for label in ENTITY_LABELS:
    ner.add_label(label)

# Create spaCy training examples
examples = []

for text, annotations in final_train_data_v7:
    doc = nlp_model4_v2.make_doc(text)
    example = Example.from_dict(doc, annotations)
    examples.append(example)

print("Training examples:", len(examples))

# Initialize model
optimizer = nlp_model4_v2.begin_training()

# Train
for iteration in range(30):

    random.shuffle(examples)
    losses = {}

    batches = minibatch(
        examples,
        size=compounding(4.0, 32.0, 1.001)
    )

    for batch in batches:
        nlp_model4_v2.update(
            batch,
            drop=0.3,
            losses=losses
        )

    print(
        f"Iteration {iteration + 1:02d} "
        f"| Loss: {losses.get('ner', 0):.4f}"
    )

print("\nModel 4 v2 training completed successfully!")

Training examples: 1653
Iteration 01 | Loss: 2189.6299
Iteration 02 | Loss: 70.9041
Iteration 03 | Loss: 76.7566
Iteration 04 | Loss: 80.4788
Iteration 05 | Loss: 63.2053
Iteration 06 | Loss: 56.2093
Iteration 07 | Loss: 70.6860
Iteration 08 | Loss: 66.3161
Iteration 09 | Loss: 64.5649
Iteration 10 | Loss: 55.7625
Iteration 11 | Loss: 66.7121
Iteration 12 | Loss: 56.5869
Iteration 13 | Loss: 72.5250
Iteration 14 | Loss: 75.5619
Iteration 15 | Loss: 73.3303
Iteration 16 | Loss: 65.4711
Iteration 17 | Loss: 75.5466
Iteration 18 | Loss: 55.7462
Iteration 19 | Loss: 71.4353
Iteration 20 | Loss: 57.6634
Iteration 21 | Loss: 66.0526
Iteration 22 | Loss: 81.7108
Iteration 23 | Loss: 82.5407
Iteration 24 | Loss: 71.7736
Iteration 25 | Loss: 52.3278
Iteration 26 | Loss: 61.9618
Iteration 27 | Loss: 65.5500
Iteration 28 | Loss: 66.0377
Iteration 29 | Loss: 57.3140
Iteration 30 | Loss: 50.3284

Model 4 v2 training completed successfully!


In [84]:
test_texts_v2 = [
    "I am a 25 year old student from Uttar Pradesh and my annual income is 120000.",
    "I am a student from Uttar Pradesh.",
    "Main 20 saal ka student hoon aur Bihar mein rehta hoon.",
    "I am a 32 year old farmer in Bihar.",
    "I belong to the OBC community and my income is 180000.",
    "Main ek divyang vyakti hoon."
]

for text in test_texts_v2:
    doc = nlp_model4_v2(text)

    print("\nInput:")
    print(text)

    print("Extracted entities:")
    for ent in doc.ents:
        print(f"{ent.text:25} → {ent.label_}")


Input:
I am a 25 year old student from Uttar Pradesh and my annual income is 120000.
Extracted entities:
25                        → AGE
student                   → OCCUPATION
Uttar Pradesh             → STATE
120000                    → ANNUAL_INCOME

Input:
I am a student from Uttar Pradesh.
Extracted entities:
student                   → OCCUPATION
Uttar Pradesh             → STATE

Input:
Main 20 saal ka student hoon aur Bihar mein rehta hoon.
Extracted entities:
20                        → AGE
student                   → OCCUPATION
Bihar                     → STATE

Input:
I am a 32 year old farmer in Bihar.
Extracted entities:
32                        → AGE
farmer                    → OCCUPATION
Bihar                     → STATE

Input:
I belong to the OBC community and my income is 180000.
Extracted entities:
OBC                       → CASTE_CATEGORY
180000                    → ANNUAL_INCOME

Input:
Main ek divyang vyakti hoon.
Extracted entities:
divyang                   → 

In [85]:
from spacy.training import Example
from spacy.scorer import Scorer

eval_examples_v2 = []

for text, annotations in test_data:
    pred_doc = nlp_model4_v2(text)
    example = Example.from_dict(pred_doc, annotations)
    eval_examples_v2.append(example)

scorer = Scorer()
scores_v2 = scorer.score(eval_examples_v2)

print("Model 4 v2 Performance")
print("----------------------")
print("Precision:", round(scores_v2["ents_p"], 4))
print("Recall:   ", round(scores_v2["ents_r"], 4))
print("F1 Score: ", round(scores_v2["ents_f"], 4))

print("\nPer-entity scores")
print("-----------------")

for label, metrics in scores_v2["ents_per_type"].items():
    precision = metrics.get("p", 0)
    recall = metrics.get("r", 0)

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0
    )

    print(
        f"{label:20} "
        f"P={precision:.4f} "
        f"R={recall:.4f} "
        f"F1={f1:.4f}"
    )

Model 4 v2 Performance
----------------------
Precision: 1.0
Recall:    0.998
F1 Score:  0.999

Per-entity scores
-----------------
CASTE_CATEGORY       P=1.0000 R=1.0000 F1=1.0000
STATE                P=1.0000 R=1.0000 F1=1.0000
AGE                  P=1.0000 R=1.0000 F1=1.0000
OCCUPATION           P=1.0000 R=0.9883 F1=0.9941
ANNUAL_INCOME        P=1.0000 R=1.0000 F1=1.0000
DISABILITY           P=1.0000 R=1.0000 F1=1.0000
GENDER               P=1.0000 R=1.0000 F1=1.0000


In [86]:
from pathlib import Path

final_model_path = Path("../models/model4_ner")

# Save the new final model
nlp_model4_v2.to_disk(final_model_path)

print("Final Model 4 saved successfully!")
print("Path:", final_model_path.resolve())

Final Model 4 saved successfully!
Path: C:\Users\hp\OneDrive\Desktop\Entity Extraction\models\model4_ner


In [87]:
import spacy

# Reload final saved model
final_model = spacy.load("../models/model4_ner")

print("Final Model 4 reloaded successfully!")
print("Pipeline:", final_model.pipe_names)
print("Labels:", final_model.get_pipe("ner").labels)

Final Model 4 reloaded successfully!
Pipeline: ['ner']
Labels: ('AGE', 'ANNUAL_INCOME', 'CASTE_CATEGORY', 'DISABILITY', 'GENDER', 'OCCUPATION', 'STATE')


In [88]:
text = """
I am a 25 year old student from Uttar Pradesh.
My annual income is 120000 and I belong to the OBC category.
"""

doc = final_model(text)

print("Input:")
print(text)

print("\nExtracted entities:")
for ent in doc.ents:
    print(f"{ent.text:20} → {ent.label_}")

Input:

I am a 25 year old student from Uttar Pradesh.
My annual income is 120000 and I belong to the OBC category.


Extracted entities:
25                   → AGE
student              → OCCUPATION
Uttar Pradesh        → STATE
120000               → ANNUAL_INCOME
OBC                  → CASTE_CATEGORY
